In [ ]:
import os
import json
import random
import cv2
from google.colab.patches import cv2_imshow
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.keras.layers import Conv2D, Concatenate, Activation
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
!pip install optuna
import optuna
!pip install optuna-integration[tfkeras]
from optuna.integration import TFKerasPruningCallback
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
import gc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install opencv-python tqdm
from tqdm import tqdm

In [ ]:
def rotate(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h),
                          borderMode=cv2.BORDER_REFLECT)

def shift(img, tx, ty):
    h, w = img.shape[:2]
    M = np.float32([[1,0,tx],[0,1,ty]])
    return cv2.warpAffine(img, M, (w,h),
                          borderMode=cv2.BORDER_REFLECT)

def zoom(img, factor):
    h, w = img.shape[:2]

    if factor > 1:
        nh, nw = int(h/factor), int(w/factor)

        y = (h-nh)//2
        x = (w-nw)//2

        crop = img[y:y+nh, x:x+nw]
        return cv2.resize(crop, (w,h))

    else:
        resized = cv2.resize(
            img,
            (int(w*factor), int(h*factor))
        )

        canvas = np.zeros_like(img)

        rh, rw = resized.shape

        y = (h-rh)//2
        x = (w-rw)//2

        canvas[y:y+rh, x:x+rw] = resized
        return canvas

def brightness(img, alpha):
    return np.clip(img.astype(np.float32)*alpha,
                   0,255).astype(np.uint8)

def clahe(img):
    cl = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )
    return cl.apply(img)

def gaussian_noise(img):
    noise = np.random.normal(
        0,
        15,
        img.shape
    )

    return np.clip(
        img.astype(np.float32)+noise,
        0,
        255
    ).astype(np.uint8)

def gaussian_blur(img):
    return cv2.GaussianBlur(img,(3,3),0)

def hflip(img):
    return cv2.flip(img,1)

def vflip(img):
    return cv2.flip(img,0)

In [ ]:
AUGS = {
    "rot_p10": lambda x: rotate(x,10),
    "rot_m10": lambda x: rotate(x,-10),

    "shift_right": lambda x: shift(x,11,0),
    "shift_left": lambda x: shift(x,-11,0),

    "shift_up": lambda x: shift(x,0,-11),
    "shift_down": lambda x: shift(x,0,11),

    "zoom_in": lambda x: zoom(x,1.10),
    "zoom_out": lambda x: zoom(x,0.90),

    "bright_p20": lambda x: brightness(x,1.2),
    "bright_m20": lambda x: brightness(x,0.8),

    "clahe": clahe,

    "gaussian_noise": gaussian_noise,

    "gaussian_blur": gaussian_blur,

    "hflip": hflip,

    "vflip": vflip
}

In [ ]:
def augment_folder(input_dir,
                   output_dir):

    os.makedirs(output_dir,
                exist_ok=True)

    imgs = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".png")
    ]

    for fname in tqdm(imgs):

        path = os.path.join(
            input_dir,
            fname
        )

        img = cv2.imread(
            path,
            cv2.IMREAD_GRAYSCALE
        )

        base = os.path.splitext(fname)[0]

        for aug_name, aug_func in AUGS.items():

            aug_img = aug_func(img)

            out_name = (
                f"{base}_{aug_name}.png"
            )

            cv2.imwrite(
                os.path.join(
                    output_dir,
                    out_name
                ),
                aug_img
            )

In [ ]:
intact_input = "/content/drive/MyDrive/rock_core_images_new/intact_hard_241831"
nonintact_input = "/content/drive/MyDrive/rock_core_images_new/non_intact_hard_241831"

intact_output = "/content/drive/MyDrive/rock_core_images_new/intact_hard_aug_241831"
nonintact_output = "/content/drive/MyDrive/rock_core_images_new/non_intact_hard_aug_241831"

augment_folder(
    intact_input,
    intact_output
)

augment_folder(
    nonintact_input,
    nonintact_output
)

In [ ]:
intact_hard_folder = "/content/drive/MyDrive/rock_core_images_new/intact_hard_aug_241831"
nonintact_hard_folder = "/content/drive/MyDrive/rock_core_images_new/non_intact_hard_aug_241831"

intact_files = set(os.listdir(intact_hard_folder))
nonintact_files = set(os.listdir(nonintact_hard_folder))

selected_files = intact_files.union(nonintact_files)

In [ ]:
selected_files = list(selected_files)
print(len(selected_files))

In [ ]:
import json

with open("/content/drive/MyDrive/rock_core_images_new/binary_241831_labels_updated (1).json", "r") as f:
    via = json.load(f)

metadata = via["_via_img_metadata"]

In [ ]:
metadata.items()

In [ ]:
label_lookup = {}

for k, v in metadata.items():
    label_lookup[v["filename"]] = {
        "rock state": v["file_attributes"]["rock state"],
        "size": v["size"]
    }

In [ ]:
label_lookup

In [ ]:
for fname_original in selected_files:

    if fname_original not in label_lookup:
        continue

    orig_label = label_lookup[fname_original]["rock state"]
    orig_size  = label_lookup[fname_original]["size"]

    base = os.path.splitext(fname_original)[0]

    for aug_name in AUGS.keys():

        new_filename = f"{base}_{aug_name}.png"

        new_key = f"{new_filename}{orig_size}"

        metadata[new_key] = {
            "filename": new_filename,
            "size": orig_size,
            "regions": [],
            "file_attributes": {
                "rock state": orig_label
            }
        }

In [ ]:
metadata

In [ ]:
sample_keys = list(via["_via_img_metadata"].keys())[-10:]

for k in sample_keys:
    print(k)

In [ ]:
len(metadata)

In [ ]:
with open("via_project_augmented.json", "w") as f:
    json.dump(
        via,
        f,
        indent=2
    )

In [ ]:
print("Total entries:",
      len(via["_via_img_metadata"]))

In [ ]:
print("Before:", len(via["_via_img_metadata"]))

added = 0

for fname_original in selected_files:

    if fname_original not in label_lookup:
        continue

    orig_label = label_lookup[fname_original]["rock state"]
    orig_size = label_lookup[fname_original]["size"]

    base = os.path.splitext(fname_original)[0]

    for aug_name in AUGS.keys():

        new_filename = f"{base}_{aug_name}.png"
        new_key = f"{new_filename}{orig_size}"

        if new_key not in metadata:

            metadata[new_key] = {
                "filename": new_filename,
                "size": orig_size,
                "regions": [],
                "file_attributes": {
                    "rock state": orig_label
                }
            }

            added += 1

print("Added:", added)
print("After:", len(via["_via_img_metadata"]))

In [ ]:
print(id(metadata))
print(id(via["_via_img_metadata"]))

In [ ]:
sample_keys = list(via["_via_img_metadata"].keys())[-10:]

for k in sample_keys:
    print(k)

In [ ]:
import os

# Assuming label_lookup, metadata, and AUGS are already defined from previous cells.
# Iterate through each original image's metadata
for fname_original, info_original in label_lookup.items():
    orig_label = info_original["rock state"]
    orig_size  = info_original["size"]
    base = os.path.splitext(fname_original)[0] # Extract base filename without extension

    # Iterate through each augmentation type
    for aug_name, _ in AUGS.items():
        # Construct the new key for the metadata dictionary
        new_key = f"{base}_{aug_name}.png{orig_size}"

        # Add the augmented image's metadata to the main metadata dictionary
        metadata[new_key] = {
            "filename": f"{base}_{aug_name}.png",
            "size": orig_size,
            "regions": [],
            "file_attributes": {
                "rock state": orig_label
            }
        }


### Re-applying Augmentation to `metadata`

It appears the previous augmentation step might not have fully updated the `metadata` dictionary. This cell re-runs the logic to ensure all augmented image entries are correctly added. This will increase the size of the `metadata` dictionary to include all original and augmented image details.

In [ ]:
import os
import re
import json

# Reload the original via data. This will ensure 'metadata' starts with the 6846 original entries.
with open("/content/drive/MyDrive/rock_core_images_new/binary_241831_labels_updated (1).json", "r") as f:
    via = json.load(f)

# 'metadata' is now a reference to the reloaded original metadata
metadata = via["_via_img_metadata"]

# Keep track of initial metadata count for reporting
initial_metadata_count = len(metadata) # This will be 6846

# This set will keep track of keys already processed to avoid adding duplicate original entries
# if an original file also happens to be in selected_files (which it shouldn't if selected_files only has augmented)
# But it's good for robustness.
processed_keys = set(metadata.keys())

# Iterate through selected_files (contains only augmented filenames from the augmented folders)
# and add them to the existing metadata dictionary.
for fname_in_folder in selected_files:
    original_base_filename = ""

    # Attempt to extract the 'ssi_XXXXXX' part from any filename
    match_id = re.search(r"ssi_(\d{6})", fname_in_folder)
    if match_id:
        ssi_id_prefix = match_id.group(0)
        original_base_filename = f"{ssi_id_prefix}.png"
    else:
        print(f"Warning: Filename '{fname_in_folder}' does not contain 'ssi_XXXXXX' pattern and cannot derive original base. Skipping.")
        continue

    if original_base_filename in label_lookup:
        info_original = label_lookup[original_base_filename]
        orig_label = info_original["rock state"]
        orig_size  = info_original["size"]

        # Construct the key for the metadata dictionary using the full filename (with augmentation suffix) and size
        metadata_key = f"{fname_in_folder}{orig_size}"

        # Add the entry to metadata only if it does not already exist
        if metadata_key not in processed_keys:
            metadata[metadata_key] = {
                "filename": fname_in_folder,
                "size": orig_size,
                "regions": [],
                "file_attributes": {
                    "rock state": orig_label
                }
            }
            processed_keys.add(metadata_key) # Add new key to set of processed keys
    else:
        print(f"Warning: Original metadata for '{original_base_filename}' (derived from '{fname_in_folder}') not found in label_lookup. Skipping.")


print(f"Initial metadata entries based on original files: {initial_metadata_count}")
print(f"New total metadata entries after adding augmented files: {len(metadata)}")

sample_keys_after_aug = list(metadata.keys())[-10:]
print("\nLast 10 ssi_ids after re-processing:")
for k in sample_keys_after_aug:
    print(k)

In [ ]:
metadata

In [ ]:
count = 0
prefix = "ssi_004250"

for key in metadata.keys():
    if key.startswith(prefix):
        count += 1

print(f"Number of entries starting with '{prefix}': {count}")

In [ ]:
len(metadata)

In [ ]:
with open(
    "via_project_augmented.json",
    "w"
) as f:
    json.dump(
        via,
        f,
        indent=2
    )

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive/rock_core_images_new/intact_hard_aug_241831"

count = sum(
    1 for f in os.listdir(folder)
    if f.lower().endswith(('.png'))
)

print("Total images:", count)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive/rock_core_images_new/non_intact_hard_aug_241831"

count = sum(
    1 for f in os.listdir(folder)
    if f.lower().endswith(('.png'))
)

print("Total images:", count)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

src = "/content/drive/MyDrive/rock_core_images_new/ssi_patches_241831"
dst = "/content/drive/MyDrive/rock_core_images_new/ssi_patches_241831_copy"

shutil.copytree(src, dst)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive/rock_core_images_new/ssi_patches_241831_copy"

count = sum(
    1 for f in os.listdir(folder)
    if f.lower().endswith(('.png'))
)

print("Total images:", count)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

src = "/content/drive/MyDrive/rock_core_images_new/intact_hard_aug_241831"
dst = "/content/drive/MyDrive/rock_core_images_new/ssi_patches_241831_copy"

os.makedirs(dst, exist_ok=True)

for file in os.listdir(src):
    if file.lower().endswith(('.png')):
        shutil.copy2(
            os.path.join(src, file),
            os.path.join(dst, file)
        )

print("Done!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

src = "/content/drive/MyDrive/rock_core_images_new/non_intact_hard_aug_241831"
dst = "/content/drive/MyDrive/rock_core_images_new/ssi_patches_241831_copy"

os.makedirs(dst, exist_ok=True)

for file in os.listdir(src):
    if file.lower().endswith(('.png')):
        shutil.copy2(
            os.path.join(src, file),
            os.path.join(dst, file)
        )

print("Done!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive/rock_core_images_new/ssi_patches_241831_copy"

count = sum(
    1 for f in os.listdir(folder)
    if f.lower().endswith(('.png'))
)

print("Total images:", count)

In [ ]:
image_folder_path = '/content/drive/MyDrive/rock core imgs'
image_folder_path_new = '/content/drive/MyDrive/rock_core_images_new'
image_folder_path

In [ ]:
data_dir = image_folder_path
data_dir_new = image_folder_path_new
BH_dir = os.listdir(data_dir)
BH_dir_new = os.listdir(data_dir_new)
print("Borehole Numbers:", BH_dir_new)

In [ ]:
counts = {c: len(os.listdir(os.path.join(data_dir, c))) for c in BH_dir}

print("\nNumber of images per borehole:")
for k, v in counts.items():
    print(f"{k}: {v}")

In [ ]:
plt.figure(figsize = (12,4))
plt.bar(counts.keys(), counts.values(), color = 'blue')
plt.xlabel("Borehole Number")
plt.ylabel("Number of Images")
plt.title("Number of Images per Borehole")

In [ ]:
fig, axes = plt.subplots(5, 3, figsize = (15, 15))

for i, c in enumerate(BH_dir):
  tray_imgs = os.listdir(os.path.join(data_dir, c))
  for j in range(3):
    BH_img_path = os.path.join(data_dir, c, random.choice(tray_imgs))
    bgr_core_img = cv2.imread(BH_img_path)
    rgb_core_img = cv2.cvtColor(bgr_core_img, cv2.COLOR_BGR2RGB)
    axes[i, j].imshow(rgb_core_img)
    axes[i, j].set_title(c)
    axes[i, j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
sizes = []
for c in BH_dir:
  for img_filename in os.listdir(os.path.join(data_dir, c))[:]:
    full_img_path = os.path.join(data_dir, c, img_filename)
    img = cv2.imread(full_img_path)

    if img is not None:
      sizes.append((img.shape[0], img.shape[1]))
    else:
      print(f"Warning: Could not load image at {full_img_path}")

if sizes:
    widths = [s[1] for s in sizes]
    heights = [s[0] for s in sizes]

    plt.figure(figsize=(8,5))
    plt.hist(widths, bins=30, alpha=0.6, label='Width')
    plt.hist(heights, bins=30, alpha=0.6, label='Height')
    plt.legend()
    plt.title("Image Size Distribution")
    plt.xlabel("Pixels")
    plt.ylabel("Frequency")
    plt.show()

    aspect_ratios = [w/h for h, w in sizes]
    plt.figure(figsize=(6,4))
    plt.hist(aspect_ratios, bins=30, color='teal')
    plt.title("Aspect Ratio Distribution (W/H)")
    plt.xlabel("Aspect Ratio")
    plt.ylabel("Count")
    plt.show()
else:
    print("No images were loaded successfully to calculate sizes and aspect ratios.")

In [ ]:
df_sizes = pd.DataFrame(sizes)
print(df_sizes.describe())

In [ ]:
sample_BH = random.choice(BH_dir)
sample_folder = os.path.join(data_dir, sample_BH)
sample_img_path = os.path.join(sample_folder, random.choice(os.listdir(sample_folder)))

sample_img = np.array(cv2.imread(sample_img_path))
sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize = (10, 5))
axes[0].imshow(sample_img)
axes[0].set_title("Sample Image")
axes[0].axis('off')

axes[1].hist(sample_img[..., 0].ravel(), bins = 256, color = 'r', alpha = 0.5, label = 'red')
axes[1].hist(sample_img[..., 1].ravel(), bins = 256, color = 'g', alpha = 0.5, label = 'green')
axes[1].hist(sample_img[..., 2].ravel(), bins = 256, color = 'b', alpha = 0.5, label = 'blue')
axes[1].set_title('Color Channel Distribution', fontsize = 12)
axes[1].set_xlabel('Pixel Value')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
means_per_borehole_channels = {bh_name: [] for bh_name in BH_dir}
stds_per_borehole_channels = {bh_name: [] for bh_name in BH_dir}

brightness = []
contrast = []
fracture_density = []

image_labels = []

for c in BH_dir:
    current_borehole_path = os.path.join(data_dir, c)
    for img_filename in os.listdir(current_borehole_path):
        full_img_path = os.path.join(current_borehole_path, img_filename)
        img = cv2.imread(full_img_path)

        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            mean_intensity = gray.mean()
            std_intensity = gray.std()
            brightness.append(mean_intensity)
            contrast.append(std_intensity)

            edges = cv2.Canny(gray, 100, 200)
            edge_density = np.sum(edges > 0) / (edges.shape[0] * edges.shape[1])
            fracture_density.append(edge_density)

            image_labels.append(f"{c}/{img_filename}")

            mean_channels = np.mean(img, axis=(0, 1))
            std_channels = np.std(img, axis=(0, 1))

            means_per_borehole_channels[c].append(mean_channels)
            stds_per_borehole_channels[c].append(std_channels)
        else:
            print(f"Warning: Could not load image at {full_img_path}")

print("\nMean pixel value and standard deviation per borehole (aggregated per channel across all images in borehole):\n")
for bh_name in BH_dir:
    if means_per_borehole_channels[bh_name]:
        avg_mean_channels = np.mean(means_per_borehole_channels[bh_name], axis=0)
        avg_std_channels = np.mean(stds_per_borehole_channels[bh_name], axis=0)

        print(f"Borehole {bh_name}:")
        print(f"Average Mean (B, G, R) = {avg_mean_channels[0]:.2f}, {avg_mean_channels[1]:.2f}, {avg_mean_channels[2]:.2f}")
        print(f"Average Std Dev (B, G, R) = {avg_std_channels[0]:.2f}, {avg_std_channels[1]:.2f}, {avg_std_channels[2]:.2f}")
    else:
        print(f"Borehole {bh_name}: No images loaded.")

plt.figure(figsize=(15, 6))
plt.plot(brightness, marker='o', linestyle='-', markersize=4)
plt.title('Brightness (Mean Intensity) for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Mean Pixel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 6))
plt.plot(contrast, marker='o', linestyle='-', markersize=4, color='orange')
plt.title('Contrast (Standard Deviation of Intensity) for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Standard Deviation of Pixel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(15, 6))
plt.plot(fracture_density, marker='o', linestyle='-', markersize=4, color='green')
plt.title('Fracture Density (Edge Density) for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Edge Density')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
R_mean = []
G_mean = []
B_mean = []
R_std  = []
G_std  = []
B_std  = []

for c in ['Hylogger_img_241831', 'Hylogger_img_262069', 'Hylogger_img_290348']:
  current_borehole_path = os.path.join(data_dir, c)
  for img_filename in os.listdir(current_borehole_path):
      full_img_path = os.path.join(current_borehole_path, img_filename)
      img = cv2.imread(full_img_path)
      img = img/255.0

      mean_R = np.mean(img[:, :, 2])
      mean_G = np.mean(img[:, :, 1])
      mean_B = np.mean(img[:, :, 0])
      std_R  = np.std(img[:, :, 2])
      std_G  = np.std(img[:, :, 1])
      std_B  = np.std(img[:, :, 0])

      R_mean.append(mean_R)
      G_mean.append(mean_G)
      B_mean.append(mean_B)
      R_std.append(std_R)
      G_std.append(std_G)
      B_std.append(std_B)


plt.figure(figsize = (12, 6))
plt.plot(R_mean, marker = 'o', markersize = 4, linestyle = '-', color = 'red')
plt.title('Mean Red Channel Intensity for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Mean Red Channel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize = (12, 6))
plt.plot(G_mean, marker = 'o', markersize = 4, linestyle = '-', color = 'green')
plt.title('Mean Green Channel Intensity for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Mean Green Channel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize = (12, 6))
plt.plot(B_mean, marker = 'o', markersize = 4, linestyle = '-', color = 'blue')
plt.title('Mean Blue Channel Intensity for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Mean Blue Channel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize = (12, 6))
plt.plot(R_std, marker = 'o', markersize = 4, linestyle = '-', color = 'red')
plt.title('Std Red Channel Intensity for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Std Red Channel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize = (12, 6))
plt.plot(G_std, marker = 'o', markersize = 4, linestyle = '-', color = 'green')
plt.title('Std Green Channel Intensity for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Std Green Channel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize = (12, 6))
plt.plot(B_std, marker = 'o', markersize = 4, linestyle = '-', color = 'blue')
plt.title('Std Blue Channel Intensity for Each Image')
plt.xlabel('Image Index')
plt.ylabel('Std Blue Channel Intensity')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mean_R_images = np.mean(R_mean, axis = 0)
mean_G_images = np.mean(G_mean, axis = 0)
mean_B_images = np.mean(B_mean, axis = 0)
std_R_images = np.mean(R_std, axis = 0)
std_G_images = np.mean(G_std, axis = 0)
std_B_images = np.mean(B_std, axis = 0)

In [ ]:
mean = np.array([mean_R_images, mean_G_images, mean_B_images])
std  = np.array([std_R_images, std_G_images, std_B_images])

In [ ]:
mean, std

In [ ]:
file_path = ('/content/drive/MyDrive/rock core imgs/Hylogger_img_290348/Hylogger_img_290348_50-55m.jfif')
loaded_img = cv2.imread(file_path)

img = loaded_img.astype(np.float32)/255.0
normalised_img = (img - mean) / std

In [ ]:
normalised_img

In [ ]:
normalised_img.min(), normalised_img.max()

In [ ]:
(normalised_img - normalised_img.min()) / (normalised_img.max() - normalised_img.min())

In [ ]:
img_rgb = cv2.cvtColor(normalised_img.astype(np.float32), cv2.COLOR_BGR2RGB)

def scale_for_display(image):
    min_val = image.min()
    max_val = image.max()
    if max_val == min_val:
        return np.zeros_like(image)
    scaled_image = (image - min_val) / (max_val - min_val)
    return scaled_image

display_img = scale_for_display(img_rgb)

plt.figure(figsize=(10, 8))
plt.imshow(display_img)
plt.title('Normalized Image (rescaled for display)')
plt.axis('off')
plt.show()

print(f"Display image min: {display_img.min():.4f}, max: {display_img.max():.4f}")

In [ ]:
mean_R_vals = []
mean_G_vals = []
mean_B_vals = []

for c in ['Hylogger_img_281139', 'Hylogger_img_290348']:
    current_borehole_path = os.path.join(data_dir, c)

    for img_filename in os.listdir(current_borehole_path):
        full_img_path = os.path.join(current_borehole_path, img_filename)
        img = cv2.imread(full_img_path)
        img = img.astype(np.float32) / 255.0

        R = img[:, :, 2]
        G = img[:, :, 1]
        B = img[:, :, 0]

        Rn = (R - mean_R_images) / std_R_images
        Gn = (G - mean_G_images) / std_G_images
        Bn = (B - mean_B_images) / std_B_images
        final_normalised_img = np.stack((Bn, Gn, Rn), axis=2)

        mean_R_vals.append(np.mean(Rn))
        mean_G_vals.append(np.mean(Gn))
        mean_B_vals.append(np.mean(Bn))
        np.save(f"norm_{img_filename}.npy", final_normalised_img)

plt.figure(figsize=(12,6))
plt.plot(mean_R_vals, 'o-', color='red')
plt.title('Normalised Mean Red Channel Intensity')
plt.xlabel('Image Index')
plt.ylabel('Normalised Mean Intensity')
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(mean_G_vals, 'o-', color='green')
plt.title('Normalised Mean Green Channel Intensity')
plt.xlabel('Image Index')
plt.ylabel('Normalised Mean Intensity')
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(mean_B_vals, 'o-', color='blue')
plt.title('Normalised Mean Blue Channel Intensity')
plt.xlabel('Image Index')
plt.ylabel('Normalised Mean Intensity')
plt.grid(True)
plt.show()

In [ ]:
img = cv2.imread("/content/drive/MyDrive/rock core imgs/Hylogger_img_241831/Hylogger_img_241831_132-138m.jfif")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

gray_blur = cv2.GaussianBlur(gray, (5,5), 0)

proj_horizontal = np.mean(gray_blur, axis=1)
proj_vertical = np.mean(gray_blur, axis = 0)

plt.figure(figsize=(10,3))
plt.plot(proj_horizontal)
plt.title("Horizontal Projection Profile")
plt.xlabel("Row index (pixels)")
plt.ylabel("Mean intensity")
plt.show()

plt.figure(figsize=(10,3))
plt.plot(proj_vertical)
plt.title("Vertical Projection Profile")
plt.xlabel("Column index (pixels)")
plt.ylabel("Mean intensity")
plt.show()

In [ ]:
plt.figure(figsize=(10, 3))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('Original Image')
plt.axis('off')
plt.show()

In [ ]:
dproj_horizontal = np.abs(np.diff(proj_horizontal))

plt.figure(figsize=(10,3))
plt.plot(dproj_horizontal)
plt.title("Absolute derivative of horizontal projection")
plt.xlabel("Row index")
plt.ylabel("|Δ intensity|")
plt.show()

In [ ]:
dproj_vertical = np.abs(np.diff(proj_vertical))

plt.figure(figsize=(10,3))
plt.plot(dproj_vertical)
plt.title("Absolute derivative of vertical projection")
plt.xlabel("Column index")
plt.ylabel("|Δ intensity|")
plt.show()

In [ ]:
jumps, props = find_peaks(dproj_horizontal, height=np.percentile(dproj_horizontal, 95), distance=400)
print(jumps)

In [ ]:
p = proj_horizontal
tray_mask = p < (0.6 * np.max(p))
ys = np.where(tray_mask)[0]
tray_top, tray_bottom = ys[0], ys[-1]

print(tray_top, tray_bottom)

In [ ]:
img_vis = img.copy()
cropped_img = img_vis[tray_top:tray_bottom, :]

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB))
plt.title('Cropped Image using tray_top and tray_bottom')
plt.axis('off')
plt.show()

In [ ]:
separator_points = [tray_top]
for j_val in jumps:
    if tray_top < j_val < tray_bottom:
        separator_points.append(j_val)
separator_points.append(tray_bottom)

separator_points = sorted(list(set(separator_points)))

rows = []
for i in range(len(separator_points) - 1):
    rows.append((separator_points[i], separator_points[i+1]))

In [ ]:
rows

In [ ]:
img_vis = img.copy()

for y1, y2 in rows:
  cv2.rectangle(img_vis, (0, y1), (img_vis.shape[1], y2), (0, 255, 0), 2)

plt.figure(figsize = (10, 5))
plt.imshow(img_vis)
plt.title('Image rows separation')
plt.axis('off')
plt.show()

In [ ]:
edges = cv2.Canny(gray_blur, 50, 150)
proj = np.mean(edges, axis = 1)

proj_smooth = cv2.GaussianBlur(proj.reshape(-1,1), (1,51), 0).ravel()
peaks, _ = find_peaks(proj_smooth, distance = 400, prominence = 5)

rows = []
for i in range(len(peaks)-1):
  rows.append((peaks[i], peaks[i+1]))

img_vis = img.copy()

for y1, y2 in rows:
  cv2.rectangle(img_vis, (0, y1), (img_vis.shape[1], y2), (255, 0, 0), 2)

plt.figure(figsize = (10, 5))
plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
plt.title('Image rows separation')
plt.axis('off')
plt.show()

In [ ]:
peaks

In [ ]:
plt.figure(figsize=(10,3))
plt.plot(proj_smooth)
plt.plot(peaks, proj_smooth[peaks], "rx")
plt.title("Detected row separators (peaks)")
plt.xlabel("Row index (pixels)")
plt.ylabel("Edge density")
plt.show()

In [ ]:
MIN_ROW_HEIGHT = 400

filtered_peaks = [peaks[0]]
for p in peaks[1:]:
    if p - filtered_peaks[-1] > MIN_ROW_HEIGHT:
        filtered_peaks.append(p)

In [ ]:
rows = []
for i in range(len(filtered_peaks) - 1):
    rows.append((filtered_peaks[i], filtered_peaks[i+1]))

In [ ]:
img_vis = img.copy()

for y1, y2 in rows:
    cv2.rectangle(img_vis, (0, y1), (img_vis.shape[1], y2), (0, 255, 0), 2)

plt.figure(figsize=(12,4))
plt.imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
plt.title("Final row separation (corrected)")
plt.axis("off")
plt.show()

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
def preprocess(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = clahe.apply(gray)

    return gray

In [ ]:
output_base_dir = '/content/drive/MyDrive/rock core images new/hylogger_241831_new'
os.makedirs(output_base_dir, exist_ok=True)

for c in ['Hylogger_img_241831']:
    current_borehole_path = os.path.join(data_dir, c)
    for img_filename in sorted(os.listdir(current_borehole_path)):
        full_img_path = os.path.join(current_borehole_path, img_filename)
        img = cv2.imread(full_img_path)

        if img is not None:
            preprocessed_img = preprocess(img)
            output_filename_png = os.path.splitext(img_filename)[0] + '.png'
            output_filepath = os.path.join(output_base_dir, output_filename_png)
            cv2.imwrite(output_filepath, preprocessed_img)
            print(f"Saved preprocessed image to: {output_filepath}")
        else:
            print(f"Warning: Could not load image {img_filename}")

In [ ]:
all_img_rows = {}

for c in ['Hylogger_img_241831']:
    current_borehole_path = os.path.join(data_dir, c)
    for img_filename in sorted(os.listdir(current_borehole_path)):
        full_img_path = os.path.join(current_borehole_path, img_filename)
        img = cv2.imread(full_img_path)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_blur = cv2.GaussianBlur(gray, (5,5), 0)

        sobel_y = cv2.Sobel(gray_blur, cv2.CV_64F, dx=0, dy=1, ksize=3)
        sobel_y = np.abs(sobel_y)

        proj_horizontal = np.mean(sobel_y, axis=1)

        proj_smooth = cv2.GaussianBlur(proj_horizontal.reshape(-1,1), (1,51), 0).ravel()

        dproj_horizontal = np.abs(np.diff(proj_smooth))

        thr = np.percentile(dproj_horizontal, 90)
        detected_separators, _ = find_peaks(dproj_horizontal, prominence=thr, distance=400)

        tray_proj_horizontal = np.mean(gray_blur, axis=1)
        tray_mask = tray_proj_horizontal < (0.6 * np.max(tray_proj_horizontal))
        ys = np.where(tray_mask)[0]
        tray_top, tray_bottom = ys[0], ys[-1]

        tray_proj_vertical = np.mean(gray_blur, axis = 0)
        tray_mask_vertical = tray_proj_vertical < (0.6 * np.max(tray_proj_vertical))
        xs = np.where(tray_mask_vertical)[0]
        tray_left, tray_right = xs[0], xs[-1]

        separator_points = [tray_top]
        for j_val in detected_separators:
            if tray_top < j_val < tray_bottom:
                separator_points.append(j_val)
        separator_points.append(tray_bottom)

        separator_points = sorted(list(set(separator_points)))

        rows = []
        for i in range(len(separator_points) - 1):
            rows.append((separator_points[i], separator_points[i+1]))

        rows_info = []
        tray_height = tray_bottom - tray_top
        tray_width = tray_right - tray_left

        for idx, (y1, y2) in enumerate(rows):
          y1_cropped = y1 - tray_top
          y2_cropped = y2 - tray_top
          row_dict = {"row_id": idx, "y1": int(y1_cropped), "y2": int(y2_cropped), "x1": 0, "x2": int(tray_width)}
          rows_info.append(row_dict)

        all_img_rows[full_img_path] = {"tray_top": int(tray_top), "tray_bottom": int(tray_bottom), "tray_left": int(tray_left), "tray_right": int(tray_right), "rows": rows_info}

        img_copy = img.copy()
        cropped_img_vis = img_copy[tray_top:tray_bottom, tray_left:tray_right]

        for y1, y2 in rows:
          y1_cropped = y1 - tray_top
          y2_cropped = y2 - tray_top
          cv2.rectangle(cropped_img_vis, (0, y1_cropped), (cropped_img_vis.shape[1], y2_cropped), (255, 0, 0), 10)

        plt.figure(figsize = (10, 5))
        plt.imshow(cv2.cvtColor(cropped_img_vis, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

In [ ]:
for c in ['Hylogger_img_241831']:
    current_borehole_path = os.path.join(data_dir, c)
    img_filenames = sorted(os.listdir(current_borehole_path))
    if img_filenames:
        img_filename = img_filenames[0]
        full_img_path = os.path.join(current_borehole_path, img_filename)
        img = cv2.imread(full_img_path)

        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            gray_blur = cv2.GaussianBlur(gray, (5,5), 0)

            sobel_y = cv2.Sobel(gray_blur, cv2.CV_64F, dx=0, dy=1, ksize=3)
            sobel_y = np.abs(sobel_y)

            proj_horizontal = np.mean(sobel_y, axis=1)

            proj_smooth = cv2.GaussianBlur(proj_horizontal.reshape(-1,1), (1,51), 0).ravel()

            dproj_horizontal = np.abs(np.diff(proj_smooth))

            thr = np.percentile(dproj_horizontal, 90)
            detected_separators, _ = find_peaks(dproj_horizontal, prominence=thr, distance=400)

            plt.figure(figsize=(15, 7))
            plt.subplot(1, 2, 1)
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.title(f'Original Image: {img_filename}')
            plt.axis('off')

            plt.subplot(1, 2, 2)
            plt.plot(dproj_horizontal)
            plt.plot(
                detected_separators,
                dproj_horizontal[detected_separators],
                'ro'
            )
            plt.title(f'Detected Separators for {img_filename}')
            plt.xlabel('Row index')
            plt.ylabel('|Δ intensity|')
            plt.show()
        else:
            print(f"Could not load image: {full_img_path}")
    else:
        print(f"No images found in directory: {current_borehole_path}")

In [ ]:
output_file_path = "/content/drive/MyDrive/rock_core_images_new/all_img_rows_241831.json"
with open(output_file_path, "r") as f:
    all_img_rows_data = json.load(f)

twenty_seventh_image_path = list(all_img_rows_data.keys())[19]
new_image_row_data = all_img_rows_data[twenty_seventh_image_path]

img = cv2.imread(twenty_seventh_image_path)

if img is None:
    print(f"Error: Could not load image at {twenty_seventh_image_path}")
else:
    print(f"Processing rows for {os.path.basename(twenty_seventh_image_path)} using data from JSON file...")

    img_copy_for_display = img.copy()
    cropped_tray_for_display = img_copy_for_display[
        new_image_row_data["tray_top"]:new_image_row_data["tray_bottom"],
        new_image_row_data["tray_left"]:new_image_row_data["tray_right"] + 1
    ]

    print(f"Extracted {len(new_image_row_data['rows'])} rows from {os.path.basename(twenty_seventh_image_path)}:")
    if len(new_image_row_data['rows']) == 0:
        print("No rows were detected for this image.")
    for row_idx, row_info in enumerate(new_image_row_data["rows"]):
        y1 = row_info["y1"]
        y2 = row_info["y2"]
        x1 = row_info["x1"]
        x2 = row_info["x2"]

        row_image = cropped_tray_for_display[y1:y2, x1:x2]

        plt.figure(figsize=(10, 3))
        plt.imshow(cv2.cvtColor(row_image, cv2.COLOR_BGR2RGB))
        plt.title(f"Row {row_idx + 1} from {os.path.basename(twenty_seventh_image_path)}")
        plt.axis('off')
        plt.show()

In [ ]:
all_img_rows

In [ ]:
all_img_rows.items()

In [ ]:
LOCAL_DIR = "ssi_patches_241831"
DRIVE_DIR = "/content/drive/MyDrive/rock core images new/ssi_patches_241831"

os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
def extract_ssi_from_row(row_img, ssi_w, ssi_h, overlap):
    patches = []
    stride_x = int(ssi_w * (1 - overlap))
    H, W = row_img.shape[:2]

    if H < ssi_h:
        row_img = cv2.resize(row_img, (W, ssi_h))

    for x in range(0, W - ssi_w + 1, stride_x):
        patch = row_img[:, x:x + ssi_w]
        patch = cv2.resize(patch, (ssi_w, ssi_h))
        patches.append(patch)

    return patches

In [ ]:
ssi_meta = []
ssi_index = 6846 ### change this next time
overlap = 0.5
ssi_w = 550
ssi_h = 550

output_base_dir = '/content/drive/MyDrive/rock core images new/hylogger_241831_new'

for original_img_path, info in all_img_rows.items():
    img_filename = os.path.basename(original_img_path)
    output_filename_png = os.path.splitext(img_filename)[0] + '.png'
    preprocessed_img_path = os.path.join(output_base_dir, output_filename_png)

    img = cv2.imread(preprocessed_img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Warning: Could not load preprocessed image at {preprocessed_img_path}")
        continue

    tray_top = info["tray_top"]
    tray_bottom = info["tray_bottom"]
    tray_left = info["tray_left"]
    tray_right = info["tray_right"]
    rows = info["rows"]

    cropped_img = img[tray_top:tray_bottom, tray_left:tray_right]

    for row in rows:
        y1, y2 = row["y1"], row["y2"]
        row_img = cropped_img[y1:y2, :]

        ssi_patches = extract_ssi_from_row(row_img, ssi_w=ssi_w, ssi_h=ssi_h, overlap=overlap)

        for patch in ssi_patches:
            fname = f"ssi_{ssi_index:06d}.png"
            drive_path = os.path.join(DRIVE_DIR, fname)
            cv2.imwrite(drive_path, patch)
            ssi_meta.append({"ssi_id": ssi_index, "image_path": original_img_path, "row_id": row["row_id"], "file": drive_path})
            ssi_index += 1

In [ ]:
meta_path = "/content/drive/MyDrive/rock core images new/ssi_241831_metadata.json"
with open(meta_path, "w") as f:
    json.dump(ssi_meta, f, indent=2)

print(f"Saved {len(ssi_meta)} SSIs")

In [ ]:
ssi_meta

In [ ]:
all_img_rows = {}

for c in ['Hylogger_img_317003']:
    current_borehole_path = os.path.join(data_dir, c)
    for img_filename in os.listdir(current_borehole_path):
        full_img_path = os.path.join(current_borehole_path, img_filename)
        img = cv2.imread(full_img_path)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_blur = cv2.GaussianBlur(gray, (5,5), 0)

        sobel_y = cv2.Sobel(gray_blur, cv2.CV_64F, dx=0, dy=1, ksize=3)
        sobel_y = np.abs(sobel_y)

        proj_horizontal = np.mean(sobel_y, axis=1)

        proj_smooth = cv2.GaussianBlur(proj_horizontal.reshape(-1,1), (1,51), 0).ravel()

        dproj_horizontal = np.abs(np.diff(proj_smooth))

        thr = np.percentile(dproj_horizontal, 90)
        detected_separators, _ = find_peaks(dproj_horizontal, prominence=thr, distance=400)

        tray_proj_horizontal = np.mean(gray_blur, axis=1)
        tray_mask = tray_proj_horizontal < (0.6 * np.max(tray_proj_horizontal))
        ys = np.where(tray_mask)[0]
        tray_top, tray_bottom = ys[0], ys[-1]

        tray_proj_vertical = np.mean(gray_blur, axis = 0)
        tray_mask_vertical = tray_proj_vertical < (0.6 * np.max(tray_proj_vertical))
        xs = np.where(tray_mask_vertical)[0]
        tray_left, tray_right = xs[0], xs[-1]

        separator_points = [tray_top]
        for j_val in detected_separators:
            if tray_top < j_val < tray_bottom:
                separator_points.append(j_val)
        separator_points.append(tray_bottom)

        separator_points = sorted(list(set(separator_points)))

        rows = []
        for i in range(len(separator_points) - 1):
            rows.append((separator_points[i], separator_points[i+1]))

        rows_info = []
        tray_height = tray_bottom - tray_top
        tray_width = tray_right - tray_left

        for idx, (y1, y2) in enumerate(rows):
          y1_cropped = y1 - tray_top
          y2_cropped = y2 - tray_top
          row_dict = {"row_id": idx, "y1": int(y1_cropped), "y2": int(y2_cropped), "x1": 0, "x2": int(tray_width)}
          rows_info.append(row_dict)

        all_img_rows[full_img_path] = {"tray_top": int(tray_top), "tray_bottom": int(tray_bottom), "tray_left": int(tray_left), "tray_right": int(tray_right), "rows": rows_info}

        img_copy = img.copy()
        cropped_img_vis = img_copy[tray_top:tray_bottom, tray_left:tray_right]

        for y1, y2 in rows:
          y1_cropped = y1 - tray_top
          y2_cropped = y2 - tray_top
          cv2.rectangle(cropped_img_vis, (0, y1_cropped), (cropped_img_vis.shape[1], y2_cropped), (255, 0, 0), 10)

        plt.figure(figsize = (10, 5))
        plt.imshow(cv2.cvtColor(cropped_img_vis, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

In [ ]:
all_img_rows = {}

for c in ['Hylogger_img_335966']:
    current_borehole_path = os.path.join(data_dir, c)
    for img_filename in os.listdir(current_borehole_path):
        full_img_path = os.path.join(current_borehole_path, img_filename)
        img = cv2.imread(full_img_path)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_blur = cv2.GaussianBlur(gray, (5,5), 0)

        sobel_y = cv2.Sobel(gray_blur, cv2.CV_64F, dx=0, dy=1, ksize=3)
        sobel_y = np.abs(sobel_y)

        proj_horizontal = np.mean(sobel_y, axis=1)

        proj_smooth = cv2.GaussianBlur(proj_horizontal.reshape(-1,1), (1,51), 0).ravel()

        dproj_horizontal = np.abs(np.diff(proj_smooth))

        thr = np.percentile(dproj_horizontal, 90)
        detected_separators, _ = find_peaks(dproj_horizontal, prominence=thr, distance=400)

        tray_proj_horizontal = np.mean(gray_blur, axis=1)
        tray_mask = tray_proj_horizontal < (0.6 * np.max(tray_proj_horizontal))
        ys = np.where(tray_mask)[0]
        tray_top, tray_bottom = ys[0], ys[-1]

        tray_proj_vertical = np.mean(gray_blur, axis = 0)
        tray_mask_vertical = tray_proj_vertical < (0.6 * np.max(tray_proj_vertical))
        xs = np.where(tray_mask_vertical)[0]
        tray_left, tray_right = xs[0], xs[-1]

        separator_points = [tray_top]
        for j_val in detected_separators:
            if tray_top < j_val < tray_bottom:
                separator_points.append(j_val)
        separator_points.append(tray_bottom)

        separator_points = sorted(list(set(separator_points)))

        rows = []
        for i in range(len(separator_points) - 1):
            rows.append((separator_points[i], separator_points[i+1]))

        rows_info = []
        tray_height = tray_bottom - tray_top
        tray_width = tray_right - tray_left

        for idx, (y1, y2) in enumerate(rows):
          y1_cropped = y1 - tray_top
          y2_cropped = y2 - tray_top
          row_dict = {"row_id": idx, "y1": int(y1_cropped), "y2": int(y2_cropped), "x1": 0, "x2": int(tray_width)}
          rows_info.append(row_dict)

        all_img_rows[full_img_path] = {"tray_top": int(tray_top), "tray_bottom": int(tray_bottom), "tray_left": int(tray_left), "tray_right": int(tray_right), "rows": rows_info}

        img_copy = img.copy()
        cropped_img_vis = img_copy[tray_top:tray_bottom, tray_left:tray_right]

        for y1, y2 in rows:
          y1_cropped = y1 - tray_top
          y2_cropped = y2 - tray_top
          cv2.rectangle(cropped_img_vis, (0, y1_cropped), (cropped_img_vis.shape[1], y2_cropped), (255, 0, 0), 10)

        plt.figure(figsize = (10, 5))
        plt.imshow(cv2.cvtColor(cropped_img_vis, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()

In [ ]:
def show_ssi_grid_from_disk(ssi_meta, start_idx, batch_size):
    end_idx = min(start_idx + batch_size, len(ssi_meta))
    batch = ssi_meta[start_idx:end_idx]

    ncols = 10
    nrows = (len(batch) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 2.2))
    axes = axes.flatten()

    for i, meta in enumerate(batch):
        img = cv2.imread(meta["file"])
        if img is None:
            continue

        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f"SSI {meta['ssi_id']}", fontsize=8)
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

In [ ]:
show_ssi_grid_from_disk(ssi_meta, start_idx=2000, batch_size=30)

In [ ]:
if os.path.exists("ssi_labels.json"):
    with open("ssi_labels.json", "r") as f:
        labels = json.load(f)
    print(f"Resuming labeling. Already labeled: {len(labels)} SSIs")
else:
    labels = {}
    print("Starting fresh labeling.")

In [ ]:
from IPython.display import clear_output

labeled_count = 0
max_labels_to_add = 30

for meta in ssi_meta:

    ssi_id = meta["ssi_id"]
    if str(ssi_id) in labels:
        continue

    if labeled_count >= max_labels_to_add:
        print(f"Labeled {max_labels_to_add} new SSIs. Stopping.")
        break

    lab = input(f"Label SSI {ssi_id} (i = intact, n = non-intact, s = skip, q = quit):").strip().lower()

    clear_output(wait=True)

    if lab == "q":
        print("Labeling interrupted by user.")
        break
    if lab == "s":
        print(f"SSI {ssi_id} skipped.")
        continue

    labels[str(ssi_id)] = 1 if lab == "i" else 0
    labeled_count += 1

    with open("ssi_labels.json", "w") as f:
        json.dump(labels, f, indent=2)

print(f"Finished labeling session. Total new SSIs labeled: {labeled_count}. Total labels overall: {len(labels)}")

In [ ]:
labels

In [ ]:
def rechange_single_label_interactively():
    global labels
    while True:
        clear_output(wait=True)
        ssi_to_relabel_input = input("Enter the SSI ID you want to rechange (e.g., '60') or 'q' to quit: ").strip()

        if ssi_to_relabel_input.lower() == 'q':
            print("Exiting re-labeling mode.")
            break

        if ssi_to_relabel_input not in labels:
            print(f"SSI ID '{ssi_to_relabel_input}' not found in your labeled data. Please try again.")
            continue

        ssi_id_int = int(ssi_to_relabel_input)
        new_label_input = input(f"Enter new label for SSI {ssi_to_relabel_input} (i = intact, n = non-intact, or 's' to skip/keep current): ").strip().lower()

        if new_label_input == 'i':
            labels[ssi_to_relabel_input] = 1
            print(f"SSI {ssi_to_relabel_input} label changed to Intact.")
        elif new_label_input == 'n':
            labels[ssi_to_relabel_input] = 0
            print(f"SSI {ssi_to_relabel_input} label changed to Non-intact.")
        elif new_label_input == 's':
            print(f"SSI {ssi_to_relabel_input} label kept as is.")
        else:
            print("Invalid input. Please enter 'i' for intact, 'n' for non-intact, or 's' to skip.")
            continue

        with open("ssi_labels.json", "w") as f:
            json.dump(labels, f, indent=2)
        print("Labels saved to ssi_labels.json.")
        print("-" * 30)

rechange_single_label_interactively()

In [ ]:
import gdown
import json

gdrive_url = 'https://drive.google.com/uc?id=1gJqbpZN5q85jW1Oxn4dmN_jck8ZprTll' # Updated to direct download link
output_filename = 'binary_241831_labels_updated.json'

gdown.download(gdrive_url, output_filename, quiet=False)

with open(output_filename, 'r') as f:
    full_json_data = json.load(f)

if "_via_img_metadata" in full_json_data:
    raw_metadata = full_json_data["_via_img_metadata"]
else:
    print("Error: '_via_img_metadata' key not found in the downloaded JSON.")
    raw_metadata = {}

first_6846_metadata = {}
count = 0
for key in sorted(raw_metadata.keys()):
    if count >= 6846:
        break
    filename = raw_metadata[key]["filename"]
    if filename.startswith('ssi_'):
        try:
            idx_str = filename.split('_')[-1].split('.')[0]
            idx = int(idx_str)

            if idx < 6846:
                first_6846_metadata[key] = raw_metadata[key]
                count += 1
        except ValueError:
            continue

output_json_data = {
    "_via_settings": full_json_data.get("_via_settings", {}),
    "_via_img_metadata": first_6846_metadata,
    "_via_attributes": full_json_data.get("_via_attributes", {})
}

with open('labels_0_6845.json', 'w') as f:
    json.dump(output_json_data, f, indent=4)

print(f"Saved {len(first_6846_metadata)} labels (first 6846 SSIs) to labels_0_6845.json")

remaining_metadata = {}
for key in sorted(raw_metadata.keys()):
    filename = raw_metadata[key]["filename"]
    if filename.startswith('ssi_'):
        try:
            idx_str = filename.split('_')[-1].split('.')[0]
            idx = int(idx_str)

            if idx >= 6846:
                remaining_metadata[key] = raw_metadata[key]
        except ValueError:
            continue

output_remaining_json_data = {
    "_via_settings": full_json_data.get("_via_settings", {}),
    "_via_img_metadata": remaining_metadata,
    "_via_attributes": full_json_data.get("_via_attributes", {})
}

with open('labels_6846_onwards.json', 'w') as f:
    json.dump(output_remaining_json_data, f, indent=4)

print(f"Saved {len(remaining_metadata)} labels (SSIs from 6846 onwards) to labels_6846_onwards.json")

In [ ]:
train_file_path = "/content/binary_241831_labels_updated.json"

classes = [
    "intact",
    "non intact"
]

class_to_index = {c:i for i,c in enumerate(classes)}

data_0_6845 = []
with open(train_file_path) as f:
    js = json.load(f)

metadata_0_6845 = js["_via_img_metadata"]

for item in metadata_0_6845.values():
    filename = item["filename"]
    label = item["file_attributes"]["rock state"]

    if label not in class_to_index:
        continue

    one_hot = np.zeros(len(classes))
    one_hot[class_to_index[label]] = 1

    data_0_6845.append([filename, label] + list(one_hot))

columns = ["filename","label"] + classes
df_0_6845 = pd.DataFrame(data_0_6845, columns=columns)

print("\n--- Data from labels_0_6845.json ---")
print(df_0_6845.head())
print("\nNumber of images in each class for labels_0_6845.json:")
print(df_0_6845['label'].value_counts())

In [ ]:
X = df_0_6845["filename"].values
y = (df_0_6845["label"] == "non intact").astype(np.float32).values
y

In [ ]:
!mkdir -p /content/ssi_241831_patches

In [ ]:
!cp -r /content/drive/MyDrive/rock_core_images_new/ssi_patches_241831-20260425T131815Z-3-001.zip .
!unzip -q ssi_patches_241831-20260425T131815Z-3-001.zip -d /content/ssi_241831_patches

In [ ]:
Local_Dir_Train = ("/content/ssi_241831_patches/ssi_patches_241831")

In [ ]:
def get_full_local_path(filename, base_dir):
    full_path = os.path.join(base_dir, filename)
    return full_path

X_full_paths = [get_full_local_path(filename, Local_Dir_Train) for filename in X]
X_path = np.array(X_full_paths)

In [ ]:
def get_full_local_path(filename, base_dir):
    full_path = os.path.join(base_dir, filename)
    return full_path

X_full_paths = [get_full_local_path(filename, Local_Dir_Train) for filename in X_test]
X_path_test = np.array(X_full_paths)

In [ ]:
X_path_test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_path, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.model_selection import train_test_split

X_val, X_test, y_val, y_test = train_test_split(
    X_val, y_val,
    test_size=0.5,
    stratify=y_val,
    random_state=42
)

In [ ]:
split_idx = int(0.8 * len(X_path)) # sequential split

X_train = X_path[:split_idx]
X_val   = X_path[split_idx:]

y_train = y[:split_idx]
y_val   = y[split_idx:]

In [ ]:
print("Train:", Counter(map(tuple, y_train)))
print("Val:", Counter(map(tuple, y_val)))

In [ ]:
X_train, X_val, X_test

In [ ]:
y_train_labels = y_train.astype(int)
class_indices = {i: np.where(y_train_labels == i)[0] for i in range(2)}

In [ ]:
y_test_labels = y_test.astype(int)
class_indices_test = {i: np.where(y_test_labels == i)[0] for i in range(2)}
y_test_labels

In [ ]:
MINORITY_CLASSES = ["non intact", "others"]

NON_INTACT_CLASS_INDEX = 1
OTHERS_CLASS_INDEX = 2

augment_flip = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
])

augment_brightness = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(0.15),
])

augment_contrast = tf.keras.Sequential([
    tf.keras.layers.RandomContrast(0.15),
])

augment_brightness_contrast = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

aug_funcs_others = (
    augment_flip,
    augment_brightness,
    augment_contrast,
    augment_brightness_contrast,
    tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomBrightness(0.15),
    ]),
    tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomContrast(0.15),
    ]),
    tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomBrightness(0.15),
        tf.keras.layers.RandomContrast(0.15),
    ]),
)

def balanced_generator(X, y_labels, class_indices, batch_size=24):
    num_classes = 3
    samples_per_class = batch_size // num_classes

    current_shuffled_indices = {c_idx: list(class_indices[c_idx]) for c_idx in range(num_classes)}

    for c_idx in range(num_classes):
        random.shuffle(current_shuffled_indices[c_idx])

    while True:
        X_batch = []
        y_batch = []

        for c_idx in range(num_classes):
            selected_indices_for_class = []

            for _ in range(samples_per_class):
                if not current_shuffled_indices[c_idx]:
                    current_shuffled_indices[c_idx] = list(class_indices[c_idx])
                    random.shuffle(current_shuffled_indices[c_idx])

                selected_indices_for_class.append(current_shuffled_indices[c_idx].pop())

            for i_global in selected_indices_for_class:
                original_img = tf.io.read_file(X[i_global])
                original_img = tf.image.decode_png(original_img, channels=1)
                original_img = tf.image.resize(original_img, (224, 224))
                original_img = tf.cast(original_img, tf.float32) / 255.0

                if c_idx == 0:
                    X_batch.append(original_img)
                    y_batch.append(y_labels[i_global])

                elif c_idx == NON_INTACT_CLASS_INDEX:
                    if np.random.rand() < 0.5:
                        augmented_img = augment_flip(tf.expand_dims(original_img, 0), training=True)
                        X_batch.append(tf.squeeze(augmented_img, axis=0))
                    else:
                        X_batch.append(original_img)
                    y_batch.append(y_labels[i_global])

                elif c_idx == OTHERS_CLASS_INDEX:
                    rand_choice = np.random.randint(0, len(aug_funcs_others) + 1)
                    if rand_choice == 0:
                        X_batch.append(original_img)
                    else:
                        chosen_augment_func = aug_funcs_others[rand_choice - 1]
                        augmented_img = chosen_augment_func(tf.expand_dims(original_img, 0), training=True)
                        X_batch.append(tf.squeeze(augmented_img, axis=0))
                    y_batch.append(y_labels[i_global])

        if not X_batch:
            continue

        X_batch_tensor = tf.stack(X_batch)
        y_batch_one_hot = tf.keras.utils.to_categorical(y_batch, num_classes)
        y_batch_tensor = tf.convert_to_tensor(y_batch_one_hot, dtype=tf.float32)

        indices_tensor = tf.constant(np.random.permutation(len(X_batch_tensor)), dtype=tf.int32)

        yield tf.gather(X_batch_tensor, indices_tensor), tf.gather(y_batch_tensor, indices_tensor)

In [ ]:
def preprocess(x, y):
    x = tf.io.read_file(x)
    x = tf.image.decode_png(x, channels=1)
    x = tf.image.resize(x, (224, 224))
    x = tf.cast(x, tf.float32) / 255.0
    y = tf.cast(y, tf.float32)

    return x, y

In [ ]:
batch_size = 32

dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
dataset = dataset.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
dataset = dataset.batch(batch_size)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))

val_dataset = val_dataset.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.batch(batch_size)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

test_dataset = test_dataset.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.batch(batch_size)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
def augment(x, y):
    x = tf.image.random_brightness(x, 0.1)
    x = tf.image.random_contrast(x, 0.9, 1.1)

    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)

    return x, y

dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
dataset, val_dataset, test_dataset

In [ ]:
y_train_labels

In [ ]:
class_indices

In [ ]:
class_counts = df_0_6845['label'].value_counts()
class_counts = np.array([
    class_counts['intact'],
    class_counts['non intact']
])
alpha = 1 / class_counts
alpha = alpha / np.sum(alpha)

In [ ]:
alpha

In [ ]:
def focal_loss_with_label_smoothing(gamma=2.0, alpha=None, label_smoothing=0.1):

    def loss(y_true, y_pred):
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)

        y_true_smoothed = y_true * (1 - label_smoothing) + label_smoothing / num_classes

        epsilon = 1e-7
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        p_t = tf.reduce_sum(y_true_smoothed * y_pred, axis=1)

        ce = -tf.math.log(p_t)

        focal_weight = tf.pow(1 - p_t, gamma)

        if alpha is not None:
            alpha_factor = tf.reduce_sum(y_true * alpha, axis=1)
            focal_weight *= alpha_factor

        loss = focal_weight * ce
        return loss

    return loss

In [ ]:
input_layer = tfl.Input(shape=(224, 224, 1))

x = tfl.Conv2D(32, (3,3), padding='same',
               kernel_initializer='he_normal', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(input_layer)
x = tfl.BatchNormalization()(x)
x = tfl.ReLU()(x)
x = tfl.MaxPooling2D((2,2))(x)

x = tfl.Conv2D(64, (3,3), padding='same',
               kernel_initializer='he_normal', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = tfl.BatchNormalization()(x)
x = tfl.ReLU()(x)
x = tfl.MaxPooling2D((2,2))(x)

x = tfl.Conv2D(128, (3,3), padding='same',
               kernel_initializer='he_normal', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = tfl.BatchNormalization()(x)
x = tfl.ReLU()(x)
x = tfl.MaxPooling2D((2,2))(x)

x = tfl.Conv2D(256, (3,3), padding='same',
               kernel_initializer='he_normal', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = tfl.BatchNormalization()(x)
x_pre_pooling = tfl.ReLU()(x)

avg_pool = tfl.GlobalAveragePooling2D()(x_pre_pooling)
max_pool = tfl.GlobalMaxPooling2D()(x_pre_pooling)

concatenated_pooling = tfl.Concatenate()([avg_pool, max_pool])

x = tfl.Dense(128, kernel_initializer='he_normal')(concatenated_pooling)
x = tfl.BatchNormalization()(x)
x = tfl.ReLU()(x)

x = tfl.Dropout(0.3)(x)

outputs = tfl.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=input_layer, outputs=outputs)

In [ ]:
def f1_macro(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=2)

    tp = tf.cast(tf.linalg.diag_part(cm), tf.float32)
    fp = tf.cast(tf.reduce_sum(cm, axis=0), tf.float32) - tp
    fn = tf.cast(tf.reduce_sum(cm, axis=1), tf.float32) - tp

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return tf.reduce_mean(f1)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss = 'binary_crossentropy',
              metrics = ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), f1_macro])

In [ ]:
model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor ='val_f1_macro',
    patience = 5,
    restore_best_weights = True,
    mode = 'max')

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor = 'val_f1_macro',
    save_best_only=True,
    mode='max')

ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau(
    monitor = 'val_f1_macro',
    factor = 0.3,
    patience = 3,
    mode = 'max')

history = model.fit(
    dataset,
    validation_data = val_dataset,
    epochs=50,
    class_weight={
        0: 1.0,
        1: 1.4
    },
    callbacks = [early_stop, checkpoint, ReduceLROnPlateau])

plt.plot(history.history['loss'], label = 'train_loss')
plt.plot(history.history['val_loss'], label = 'validation_loss')

if 'val_f1_macro' in history.history:
    plt.plot(history.history['val_f1_macro'], label='validation_f1_macro')

plt.legend()
plt.show()

In [ ]:
model.save_weights('best_model.weights.h5')

In [ ]:
y_true = []
y_pred = []

for x_batch, y_batch in val_dataset:
    preds = model.predict(x_batch, verbose=0)

    y_true.extend(y_batch.numpy().astype(int))
    y_pred.extend((preds > 0.5).astype(int).flatten())

In [ ]:
y_true_test = []
y_pred_test = []

for x_batch, y_batch in test_dataset:
    preds = model.predict(x_batch, verbose=0)

    y_true_test.extend(y_batch.numpy().astype(int))
    y_pred_test.extend((preds>0.5).astype(int).flatten())

In [ ]:
predictions = []
confidences = []

for x_batch in val_dataset:
    preds = model.predict(x_batch, verbose=0)

    predictions.extend(np.argmax(preds, axis=1))
    confidences.extend(np.max(preds, axis=1))

In [ ]:
predictions = []
confidences = []

for x_batch in test_dataset:
    preds = model.predict(x_batch, verbose=0)

    predictions.extend(np.argmax(preds, axis=1))
    confidences.extend(np.max(preds, axis=1))

In [ ]:
import os
import json

results = {}

idx = 0

for x_batch in test_dataset:
    preds = model.predict(x_batch, verbose=0)

    pred_classes = np.argmax(preds, axis=1)
    confs = np.max(preds, axis=1)

    batch_size = len(pred_classes)

    for i in range(batch_size):
        file_path = X_test[idx]
        file_name = os.path.basename(file_path)

        results[file_name] = {
            "label": classes[pred_classes[i]],
            "confidence": float(confs[i])
        }

        idx += 1

In [ ]:
with open("predictions.json", "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
import json

with open("ssi_patches_241831_mid_3100.json", "r") as f:
    patches = json.load(f)

with open("predictions_3100.json", "r") as f:
    preds = json.load(f)

for key, entry in patches["_via_img_metadata"].items():
    filename = entry["filename"]
    if filename in preds:
        entry["file_attributes"]["rock state"] = preds[filename]["label"]

with open("ssi_patches_updated.json", "w") as f:
    json.dump(patches, f, indent=4)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

classes = ['intact', 'non intact']
print(classification_report(y_true, y_pred, target_names=classes, digits=2))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

classes = ['intact', 'non intact']
print(classification_report(y_true_test, y_pred_test, target_names=classes, digits=2))

In [ ]:
import random

sample_ssi_filename = random.choice(os.listdir(Local_Dir_Train))
sample_input_image_path = os.path.join(Local_Dir_Train, sample_ssi_filename)

original_img_for_display = cv2.imread(sample_input_image_path)
img = cv2.imread(sample_input_image_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    print(f"Error: Could not load image at {sample_input_image_path}")
else:
    img = cv2.resize(img, (224, 224))
    img = img.astype(np.float32) / 255.0

    sample_input_image = img[np.newaxis, :, :, np.newaxis]

    layer_outputs = []
    layer_names = []
    for layer in model.layers:
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.MaxPooling2D)):
            layer_outputs.append(model.get_layer(layer.name).output)
            layer_names.append(layer.name)

    activation_model = tf.keras.models.Model(inputs=model.input, outputs=layer_outputs)

    feature_maps = activation_model.predict(sample_input_image, verbose=0)

    for i, fmap in enumerate(feature_maps):
        if fmap.ndim == 4:
            fmap_display = fmap[0]

            num_channels = fmap_display.shape[-1]
            print(f"Layer: {layer_names[i]}, Feature Map Shape: {fmap_display.shape}")

            num_plots = min(num_channels, 8)
            if num_plots == 0:
                continue

            fig, axes = plt.subplots(1, num_plots, figsize=(num_plots * 2.5, 3))
            fig.suptitle(f"Feature Maps for Layer: {layer_names[i]}", fontsize=16)

            if num_plots == 1:
                axes = [axes]

            for j in range(num_plots):
                ax = axes[j]
                channel_map = fmap_display[:, :, j]

                min_val = np.min(channel_map)
                max_val = np.max(channel_map)
                if max_val - min_val > 0:
                    channel_map_normalized = (channel_map - min_val) / (max_val - min_val)
                else:
                    channel_map_normalized = np.zeros_like(channel_map)

                ax.imshow(channel_map_normalized, cmap='viridis')
                ax.set_title(f'Channel {j}')
                ax.axis('off')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        else:
            print(f"Skipping layer {layer_names[i]} due to incompatible dimensions: {fmap.shape}")

    plt.figure(figsize=(8,8))
    plt.imshow(cv2.cvtColor(original_img_for_display, cv2.COLOR_BGR2RGB))
    plt.title(f'Original Input Image: {sample_ssi_filename}')
    plt.axis('off')
    plt.show()

In [ ]:
import random
import math
import os

def display_mislabeled_images_for_split(dataset, original_filenames_array, true_labels_list, pred_labels_list, split_name, max_images_to_display=None):

    mislabeled_indices_global = [i for i, (t, p) in enumerate(zip(true_labels_list, pred_labels_list)) if t != p]

    if not mislabeled_indices_global:
        print(f"No mislabeled images found in {split_name} set.")
        return

    # If max_images_to_display is None, display all mislabeled images
    if max_images_to_display is None:
        display_limit = len(mislabeled_indices_global)
    else:
        display_limit = min(max_images_to_display, len(mislabeled_indices_global))

    print(f"Displaying up to {display_limit} mislabeled images from {split_name} set:")

    images_to_plot = []
    ssi_ids_to_plot = []
    true_labels_to_plot = []
    pred_labels_to_plot = []

    global_count = 0
    for x_batch, _ in dataset:
        batch_size_current = x_batch.shape[0]
        for i_in_batch in range(batch_size_current):
            if global_count in mislabeled_indices_global:
                if len(images_to_plot) < display_limit:
                    images_to_plot.append(x_batch[i_in_batch].numpy().squeeze())
                    true_labels_to_plot.append(true_labels_list[global_count])
                    pred_labels_to_plot.append(pred_labels_list[global_count])

                    filename = original_filenames_array[global_count]
                    ssi_name = os.path.basename(filename)
                    ssi_id_str = ssi_name.replace('ssi_', '').replace('.png', '')
                    try:
                        ssi_ids_to_plot.append(int(ssi_id_str))
                    except ValueError:
                        ssi_ids_to_plot.append("N/A")
                else:
                    break

            global_count += 1
        if len(images_to_plot) >= display_limit:
            break

    num_images = len(images_to_plot)
    if num_images == 0:
        print(f"No mislabeled images found in {split_name} set (after filtering for max_images_to_display).")
        return

    ncols = 5
    nrows = math.ceil(num_images / ncols)

    plt.figure(figsize=(15, nrows * 3))

    for i in range(num_images):
        plt.subplot(nrows, ncols, i + 1)
        plt.imshow(images_to_plot[i], cmap='gray')
        plt.title(f"SSI: {ssi_ids_to_plot[i]}\nTrue: {classes[true_labels_to_plot[i]]}\nPred: {classes[pred_labels_to_plot[i]]}", fontsize=10)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

display_mislabeled_images_for_split(val_dataset, X_val, y_true, y_pred, "Validation", max_images_to_display=len(mislabeled_indices_global) if 'mislabeled_indices_global' in locals() else None)

display_mislabeled_images_for_split(test_dataset, X_test, y_true_test, y_pred_test, "Test", max_images_to_display=len(mislabeled_indices_global) if 'mislabeled_indices_global' in locals() else None)

In [ ]:
def build_model(trial):

    filters_base = trial.suggest_categorical("filters_base", [16, 32, 64])
    num_blocks = trial.suggest_int("num_blocks", 3, 5)
    use_attention = trial.suggest_categorical("use_attention", [True, False])
    attention_start = trial.suggest_int("attention_start", 0, num_blocks-1)
    dense_units = trial.suggest_categorical("dense_units", [64, 128, 256])
    dropout1 = trial.suggest_float("dropout1", 0.2, 0.4)
    dropout2 = trial.suggest_float("dropout2", 0.3, 0.6)

    inputs = tf.keras.Input(shape=(224,224,3))
    x = inputs

    filters = filters_base

    for i in range(num_blocks):
        kernel_size = trial.suggest_categorical(
        f"kernel_block_{i}", [3, 5, 7]
       )
        growth = trial.suggest_categorical(f"growth_{i}", [1.5, 2])
        x = tfl.Conv2D(filters, kernel_size, padding='same')(x)
        x = tfl.BatchNormalization()(x)
        x = tfl.ReLU()(x)

        if use_attention and i >= attention_start:
            x = SpatialAttention()(x)

        x = tfl.MaxPooling2D()(x)

        filters = int(filters * growth)
        filters = min(filters, 512)

    x = tfl.Dropout(dropout1)(x)
    x = tfl.GlobalAveragePooling2D()(x)
    x = tfl.Dense(dense_units, activation='relu')(x)
    x = tfl.BatchNormalization()(x)
    x = tfl.Dropout(dropout2)(x)

    outputs = tfl.Dense(6, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    return model

In [ ]:
def objective(trial):

    model = build_model(trial)

    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    gamma = trial.suggest_float("gamma", 1.0, 3.0)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss = focal_loss_with_label_smoothing(gamma, alpha=alpha, label_smoothing=0.1),
        metrics = ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.F1Score(average="macro")]
        )

    early_stop = tf.keras.callbacks.EarlyStopping(
    monitor ='val_loss',
    patience = 5,
    restore_best_weights = True)

    checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor = 'val_loss',
    save_best_only=True)

    ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.3,
    patience = 3)

    Pruning = TFKerasPruningCallback(trial, "val_loss")

    history = model.fit(
        dataset,
        validation_data=val_dataset,
        epochs = 5,
        steps_per_epoch=len(X_train) // batch_size,
        verbose=0,
        callbacks=[early_stop, checkpoint, ReduceLROnPlateau, Pruning]
    )

    return max(history.history['val_f1_score'])

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

In [ ]:
print("Best params:", study.best_params)

In [ ]:
best_trial = study.best_trial

model = build_model(best_trial)

model.compile(
    optimizer=tf.keras.optimizers.Adam(best_trial.params["lr"]),
    loss = focal_loss_with_label_smoothing(best_trial.params["gamma"], alpha=alpha, label_smoothing=0.1),
    metrics = ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.F1Score()]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor ='val_loss',
    patience = 5,
    restore_best_weights = True)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor = 'val_loss',
    save_best_only=True)

ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau(
    monitor = 'val_loss',
    factor = 0.3,
    patience = 3)

model.fit(
    dataset,
    validation_data=val_dataset,
    epochs=25,
    steps_per_epoch=len(X_train) // batch_size,
    callbacks=[early_stop, checkpoint, ReduceLROnPlateau]
)

plt.plot(history.history['loss'], label = 'train_loss')
plt.plot(history.history['val_loss'], label = 'validation_loss')
plt.plot(history.history['val_f1_score'], label = 'validation_f1')
plt.legend()
plt.show()

In [ ]:
import json


# ==========================================
# LOAD JSON FILE
# ==========================================

json_path = "/content/ssi_patches_241831_end_6845_updated.json"

with open(json_path, "r") as f:
    data = json.load(f)


# ==========================================
# CHANGE LABELS
# others --> non intact
# ==========================================

count = 0

# Corrected loop to access image metadata and update 'rock state' in 'file_attributes'
if "_via_img_metadata" in data:
    for image_key in data["_via_img_metadata"]:
        file_attrs = data["_via_img_metadata"][image_key].get("file_attributes", {})

        if file_attrs.get("rock state") == "others":
            file_attrs["rock state"] = "non intact"
            count += 1


# ==========================================
# SAVE UPDATED FILE
# ==========================================

output_path = "binary_241831_labels_updated.json"

with open(output_path, "w") as f:
    json.dump(data, f, indent=2)


print(f"Converted {count} labels.")
print(f"Saved updated file to: {output_path}")